# 7장 — 청킹과 검색 품질 🔪

희석의 수학을 코드로 확인하고, recall@k 채점기를 직접 만들고, 평가 세트 크기의 함정까지 봅니다.

**이 노트북의 목표**
1. 희석: 여러 주제를 평균한 벡터는 어느 질문과도 코사인이 낮다 (통문서의 저주)
2. recall@k 채점기를 만들어 k에 따른 성적 변화를 확인
3. 평가 세트가 작으면 recall 측정값이 얼마나 출렁이는지 (3부 8장 재회)

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch7')

## 1. 실험 ① — 희석의 수학: 통문서는 왜 뜨뜻미지근한가

toy 임베딩: 주제 4개(결혼·장례·출산·절차)를 서로 직교하는 방향으로 둡니다.
통문서 벡터 = 조항 벡터들의 평균.

In [ ]:
import numpy as np

def normalize(v):
    return v / np.linalg.norm(v)

# 주제 축 4개 = 표준 기저 (서로 직교하는 뜻)
clauses = {
    '제1조(결혼)': np.array([1.0, 0, 0, 0]),
    '제2조(장례)': np.array([0, 1.0, 0, 0]),
    '제3조(출산)': np.array([0, 0, 1.0, 0]),
    '제4조(절차)': np.array([0, 0, 0, 1.0]),
}
whole_doc = normalize(sum(clauses.values()))     # 통문서 = 조항들의 평균 방향
q = normalize(np.array([0.97, 0.1, 0.1, 0.15]))  # 질문 "결혼 축하금 얼마?" (결혼 방향 + 약간의 잡음)

print(f'질문 · 결혼 조항  = {q @ clauses["제1조(결혼)"]:.2f}   ← 조각이 뾰족하게 맞는다')
print(f'질문 · 통문서     = {q @ whole_doc:.2f}   ← 4개 주제의 평균이라 희석')
print(f'(균등 혼합의 이론값: 1/√4 = {1/np.sqrt(4):.2f} 부근)')

assert q @ clauses['제1조(결혼)'] > 0.9, '딱 맞는 조각과는 코사인이 높아야'
assert q @ whole_doc < 0.7, '통문서(평균 방향)와는 확 낮아져야 — 희석'
print('\n✔ 확인: 주제 4개를 평균하면 코사인이 1/√4 수준으로 떨어진다 — 통문서의 저주')

## 2. 실험 ② — recall@k 채점기

평가 세트 = (질문, 정답 문서) 쌍. 검색 결과에서 정답이 몇 위였는지만 있으면 채점할 수 있습니다.

In [ ]:
# 웹 실험과 같은 평가 결과 — 각 질문에서 정답 문서의 검색 순위
ranks = [1, 3, 2, 6, 1]

def recall_at_k(ranks, k):
    """top-k 안에 정답이 들어온 질문의 비율."""
    return sum(r <= k for r in ranks) / len(ranks)

print(' k    recall@k')
for k in [1, 2, 3, 5, 10]:
    print(f'  {k:2d}   {recall_at_k(ranks, k):5.0%}')

assert recall_at_k(ranks, 1) == 0.4
assert recall_at_k(ranks, 3) == 0.8
assert recall_at_k(ranks, 5) == 0.8, 'k를 5로 늘려도 6위 문항은 못 구한다'
print('\n✔ 확인: k↑ → recall↑ 이지만 한계가 있다 — 순위 6위 문항은 청킹·임베딩을 고쳐야 할 신호')

## 3. 실험 ③ — 평가 세트는 몇 문항? (3부 8장 재회)

진짜 실력이 recall 80%인 시스템을 n문항짜리 평가 세트로 측정하는 상황을 시뮬레이션합니다.
측정값이 얼마나 출렁이는지 보세요.

In [ ]:
rng = np.random.default_rng(77)
TRUE_RECALL = 0.8

print(' 문항 수 n   측정값의 범위 (100회 반복)    이론 오차 √(p(1-p)/n)')
stds = {}
for n in [5, 20, 200]:
    measures = rng.binomial(n, TRUE_RECALL, size=100) / n
    stds[n] = measures.std()
    theory = np.sqrt(TRUE_RECALL * (1 - TRUE_RECALL) / n)
    print(f'   {n:4d}     {measures.min():4.0%} ~ {measures.max():4.0%} (σ={measures.std():.3f})      {theory:.3f}')

assert stds[5] > stds[20] > stds[200], '문항이 많을수록 측정이 안정된다 (오차 ~ 1/√n)'
assert stds[5] > 0.1, '5문항짜리 평가는 ±10%p 이상 출렁인다 — 개선 판정에 쓰기 위험'
print('\n✔ 확인: 5문항 recall의 60%→80%는 우연일 수 있다 — 수십~수백 문항을 모을 것 (3부 8장)')

## 4. 관찰 정리 + 🤖 AI 연결

- **희석**: 여러 주제의 평균 벡터는 어느 질문과도 어중간(균등 혼합이면 ≈ 1/√주제 수) —
  통문서 대신 뜻이 완결되는 조각(조항·문단)으로. 너무 잘면 조건이 찢어진다(겹침·제목 붙이기로 보완).
- **recall@k** = 평가 세트에서 top-k 안에 정답이 든 비율 — 검색 개선의 공식 성적표.
  k는 아깝게 놓친 정답만 구한다 — 순위가 낮으면 청킹·임베딩(검색 앞단)의 몫.
- **평가 세트 크기**: recall은 표본 비율이라 오차 ~ 1/√n — 5문항 성적표는 여론조사 5명과 같다.
- 정신은 6부 9장 그대로: **바꿨으면 재라.**

**다음 챕터**: 8장 — RAG 조립 (핵심). 검색이 프롬프트가 되고 답이 되는 전체 조립 +
실패 모드 진단 + 조련 vs RAG 실무 선택 기준.